# Ансамбли и случайный лес: пример на Python

Сравним дерево и лес на 300 синтетических заказах 3D-печати. Признаки известны до печати: высота, число цветов, объём пластика. Ответ — время в минутах.

Запускайте ноутбук сверху вниз из папки `09-random-forest` в копии репозитория. Нужен соседний файл `../07-features-regularization/data/train.csv`. Данные придуманы для учебника.


In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.tree import DecisionTreeRegressor

data = pd.read_csv("../07-features-regularization/data/train.csv")
X = data[["height_cm", "colors", "plastic_cm3"]]
y = data["minutes"]


## Сравнение моделей

Каждая треть таблицы по очереди служит для проверки, остальные две — для обучения. Обе модели получают одинаковые разбиения. Бутстрап-выборки леса строятся только из обучающей части.

`cross_val_score` возвращает отрицательную MAE; меняем знак, чтобы получить обычную ошибку в минутах.


In [ ]:
folds = KFold(n_splits=3, shuffle=True, random_state=42)
models = {
    "Дерево": DecisionTreeRegressor(random_state=0),
    "Лес": RandomForestRegressor(
        n_estimators=100, max_features=2,
        bootstrap=True, random_state=0,
    ),
}

for name, model in models.items():
    errors = -cross_val_score(
        model, X, y, cv=folds, scoring="neg_mean_absolute_error"
    )
    print(name, [round(error, 2) for error in errors],
          round(errors.mean(), 2))


При указанных настройках дерево даёт ошибки 3,02; 2,97; 2,69 минуты, лес — 2,76; 2,48; 2,55 минуты. Средние MAE равны 2,89 и 2,60 минуты соответственно.

Это сравнение двух моделей на одном учебном наборе. Если выбирать настройки по этим результатам, для независимой итоговой оценки понадобятся отдельные данные.
